# 의약품 목록 분류 RNN 모형 튜닝

| 변수명 | 타입 | 설명 |
| -- | -- | -- |
| 금기유형 | 범주 | 병용금기 / 임부금기 / 연령금기 / 노인주의 / 노인주의_해열진통소염제 |
| 성분코드 | 문자 | 성분 식별 코드 (7번째 글자 = 투여경로: A 내복·B 주사·C 외용·D 기타) |
| 성분명 | 텍스트 | 주 성분 영문명 (예: cyclosporine) |
| 병용성분코드 | 문자 | 함께 쓰면 금기인 병용 성분 코드 |
| 병용성분명 | 텍스트 | 병용 성분 영문명 |
| 금기등급 | 범주 | 임부금기 등급 1 / 2 / M |
| 특정연령 | float | 연령금기 기준 연령 |
| 특정연령단위 | 범주 | 개월/세 등 |
| 연령처리조건 | 범주 | 연령 조건 (이상/미만 등) |
| 상세정보 | 텍스트 | 금기 사유·위험 설명 (텍스트 분석·임베딩·LDA 대상) |
| 비고 | 텍스트 | 추가 참고사항 |
| 고시번호 | float | 금기 고시 번호 |
| 고시일자 | 날짜 | 금기 고시일 (2004-01-16 ~ 2025-12-23) |
| 제품수 | int | 해당 금기에 해당하는 의약품 제품 수 (1~11,252) |

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.express as px

In [9]:
dur = pd.read_csv('/content/의약품안전사용서비스(DUR)_통합_성분단위_2026.6.csv', encoding='utf-8-sig')
print(dur.shape)
dur.head(3)

(28502, 14)


,금기유형,성분코드,성분명,병용성분코드,병용성분명,금기등급,특정연령,특정연령단위,연령처리조건,상세정보,비고,고시번호,고시일자,제품수
0,병용금기,139230BIJ,cyclosporine,454002ATB,rosuvastatin calcium (as rosuvastatin),NaN,NaN,NaN,NaN,혈중농도 상승 및 근육병증/횡문근융해증 위험성 증가,NaN,20200061.0,2020-07-10,180
1,병용금기,157501ATR,felodipine,179101ACH,itraconazole,NaN,NaN,NaN,NaN,"Felodipine의 혈중농도 증가. 저혈압, 말초부종 같은 약물이상반응 발생 위험...",NaN,20190131.0,2019-12-17,272
2,병용금기,179101ATB,itraconazole,633800ATB,ezetimibe | atorvastatin calcium(as atorvastatin),NaN,NaN,NaN,NaN,"근육병증, 횡문근융해증 위험 증가 가능",NaN,20230083.0,2023-12-20,11252


**[미션 풀이]**

1. DUR 데이터를 활용하여 [상세정보]를 입력했을 때, [임부금기] 1등급 여부를 판별하는 모델을 구성하고, 모형의 파라미터를 바꾸면 성능이 어떻게 변화하는지 학습하시오.
    - RNN 계열의 알고리즘을 중 하나를 선택하여 학습
    - Node수 8개 / 32개 / 128개 일 때, 각각 모형의 파라미터 수, 학습 시간, 학습 정확도, 검증 정확도, F1 스코어를 계산하시오.
    - Layer수 1층 / 2층 / 3층 일 때, (노드는 32개로 고정) 각각 모형의 파라미터 수, 학습 시간, 학습 정확도, 검증 정확도, F1 스코어를 계산하시오.
    - 모형의 평가 결과를 해석하시오.

In [10]:
# 1. 데이터 준비 — 임부금기 1·2등급만 사용
im = dur[dur['금기유형'] == '임부금기'].copy()

im['금기등급'] = im['금기등급'].astype(str)
im = im[im['금기등급'].isin(['1', '2'])].copy()    # M등급·결측 제외
im['y'] = (im['금기등급'] == '1').astype(int)      # 1등급 = 1, 2등급 = 0
print('데이터:', len(im), '/ 1등급 비율:', round(im['y'].mean(), 3))

데이터: 3970 / 1등급 비율: 0.161


In [14]:
im

,금기유형,성분코드,성분명,병용성분코드,병용성분명,금기등급,특정연령,특정연령단위,연령처리조건,상세정보,비고,고시번호,고시일자,제품수,y
23639,임부금기,100301ACH,Chenodeoxycholic acid + Ursodeoxycholic acid,NaN,NaN,1,NaN,NaN,NaN,"임부 투여금기, 동물실험에서 간독성의 증거 나타남",NaN,20200130.0,2020-12-28,1,1
23640,임부금기,100501ATE,acamprosate,NaN,NaN,2,NaN,NaN,NaN,임부에 대한 안전성 미확립.,NaN,20160155.0,2016-12-30,2,0
23641,임부금기,100601ATB,acarbose,NaN,NaN,2,NaN,NaN,NaN,임부에 대한 안전성 미확립.,NaN,20160155.0,2016-12-30,1,0
23642,임부금기,100602ATB,acarbose,NaN,NaN,2,NaN,NaN,NaN,임부에 대한 안전성 미확립.,NaN,20160155.0,2016-12-30,1,0
23643,임부금기,100901ACH,aceclofenac,NaN,NaN,2,NaN,NaN,NaN,임부에 대한 안전성 미확립.임신 말기에 투여시 태아의 동맥관조기폐쇄 가능성.동물실험...,NaN,20160155.0,2016-12-30,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27644,임부금기,F00900ACH,pseudoephedrine,NaN,NaN,2,NaN,NaN,NaN,임부에 대한 안전성 미확립.,NaN,20200061.0,2020-07-10,1,0
27645,임부금기,F01100ATB,retinol,NaN,NaN,1,NaN,NaN,NaN,"임신초기 3개월간 고용량 투여시 최기형 가능성. (단, VitA(레티놀)로서 500...",NaN,20160155.0,2016-12-30,1,1
27646,임부금기,F01200ACS,guaifenesin + ibuprofen + pseudoephedrine,NaN,NaN,2,NaN,NaN,NaN,guaifenesin: - / ibuprofen: 임부에 대한 안전성 미확립. 임신...,NaN,20160155.0,2016-12-30,1,0
27647,임부금기,F01600ACS,guaifenesin + pseudoephedrine,NaN,NaN,2,NaN,NaN,NaN,guaifenesin: - / pseudoephedrine: 임부에 대한 안전성 미확립.,NaN,20200061.0,2020-07-10,1,0


In [17]:
# %pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 32.1 MB/s eta 0:00:00


In [18]:
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense, Dropout

In [19]:
from konlpy.tag import Okt

# 2. 불용어 로드 + 형태소 분석
stopwords = [w.strip() for w in open('stopword.txt', encoding='utf-8') if w.strip()]
if stopwords[0] == '불용어':
    stopwords = stopwords[1:]
print('불용어 수:', len(stopwords))

okt = Okt()
im['명사'] = [' '.join([n for n in okt.nouns(str(t))
                      if len(n) > 1 and n not in stopwords])
             for t in im['상세정보'].fillna('')]
im[['상세정보', '명사']].head(3)

불용어 수: 66


,상세정보,명사
23639,"임부 투여금기, 동물실험에서 간독성의 증거 나타남",임부 투여 금기 동물실험 간독성 증거 나타남
23640,임부에 대한 안전성 미확립.,임부 안전성 확립
23641,임부에 대한 안전성 미확립.,임부 안전성 확립


In [30]:
# 학습/검증 분리
X_train, X_test, Y_train, Y_test = train_test_split(im['명사'].values, im['y'].values, test_size=0.2, stratify=im['y'], random_state=42)


In [31]:
# 정수 시퀀스 → 패딩
tok = Tokenizer(oov_token='OOV')
tok.fit_on_texts(X_train)               # 학습 데이터로만 단어 사전 구성
vocab = len(tok.word_index) + 1         # 단어수

seq_train = tok.texts_to_sequences(X_train)
seq_test = tok.texts_to_sequences(X_test)
max_len = int(np.percentile([len(s) for s in seq_train], 95))   # 95% 분위수로 길이 통일

X_train_vec = pad_sequences(seq_train, maxlen=max_len, padding='post', truncating='post')
X_test_vec = pad_sequences(seq_test, maxlen=max_len, padding='post', truncating='post')
print('학습:', X_train.shape, '/ 검증:', X_test.shape, '/ 단어수:', vocab, '/ maxlen:', max_len)

학습: (3176,) / 검증: (794,) / 단어수: 1133 / maxlen: 48


In [32]:
# 평가 함수
from sklearn.metrics import classification_report, accuracy_score, f1_score
import plotly.express as px

def threshold_func(result):
    if result >= 0.5:
        return 1
    return 0

def Eval_func(model, history):
    df_hist = pd.DataFrame(history.history)
    Y_train_pred = pd.Series(model.predict(X_train_vec).flatten()).apply(threshold_func)
    Y_test_pred  = pd.Series(model.predict(X_test_vec).flatten()).apply(threshold_func)
    print(classification_report(Y_train, Y_train_pred))
    print(classification_report(Y_test, Y_test_pred))
    fig1 = px.line(df_hist, y=['loss','val_loss'], markers=True, title='모형 학습 / 검증 손실')
    fig1.update_yaxes(range=[0, 1])
    return fig1

In [46]:
# 실험 결과를 표로 모으기 위한 기록용 함수
results = []

def record(name, nodes, layers, model, train_sec):
    Y_train_pred = pd.Series(model.predict(X_train_vec, verbose=0).flatten()).apply(threshold_func)
    Y_test_pred  = pd.Series(model.predict(X_test_vec, verbose=0).flatten()).apply(threshold_func)
    results.append({
        '조건': name,
        '노드수': nodes,
        '층수': layers,
        '파라미터수': model.count_params(),
        '학습시간(초)': round(train_sec, 1),
        '학습정확도': round(accuracy_score(Y_train, Y_train_pred), 3),
        '검증정확도': round(accuracy_score(Y_test, Y_test_pred), 3),
        'F1 스코어': round(f1_score(Y_test, Y_test_pred), 3),
    })
    print(results[-1])

In [34]:
import tensorflow as tf

# 학습 모델의 결과를 똑같이 재현할 수 있도록 난수를 고정
tf.config.experimental.enable_op_determinism()
tf.keras.utils.set_random_seed(1234)


In [55]:
# 실험 1) 노드 8개 / 1층
tf.keras.utils.set_random_seed(1234)

model_RNN1 = Sequential()
model_RNN1.add(Input(shape=(max_len,)))
model_RNN1.add(Embedding(vocab, 64))
model_RNN1.add(SimpleRNN(8))                   # 노드 8개
model_RNN1.add(Dropout(0.5))
model_RNN1.add(Dense(1, activation='sigmoid'))
model_RNN1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_RNN1.summary()

start = time.time()
history_RNN1 = model_RNN1.fit(X_train_vec, Y_train, epochs=10, batch_size=32,
                              validation_data=(X_test_vec, Y_test),
                              class_weight={0: 1, 1: 5}, verbose=0)
time1 = time.time() - start

record('노드 8개', 8, 1, model_RNN1, time1)
Eval_func(model_RNN1, history_RNN1)

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_15 (Embedding)        │ (None, 48, 64)         │        72,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_22 (SimpleRNN)       │ (None, 8)              │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 73,105 (285.57 KB)

 Trainable params: 73,105 (285.57 KB)

 Non-trainable params: 0 (0.00 B)

{'조건': '노드 8개', '노드수': 8, '층수': 1, '파라미터수': 73105, '학습시간(초)': 42.2, '학습정확도': 0.91, '검증정확도': 0.882, 'F1 스코어': 0.706}
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.99      0.91      0.94      2666
           1       0.65      0.93      0.77       510

    accuracy                           0.91      3176
   macro avg       0.82      0.92      0.86      3176
weighted avg       0.93      0.91      0.92      3176

              precision    recall  f1-score   support

           0       0.98      0.88      0.93       666
           1       0.59      0.88      0.71       128

    accuracy                           0.88       794
   macro avg       0.78      0.88      0.82       794
weighted avg       0.91      0.88      0.89       794



In [56]:
# 실험 2) 노드 32개 / 1층
tf.keras.utils.set_random_seed(1234)

model_RNN2 = Sequential()
model_RNN2.add(Input(shape=(max_len,)))
model_RNN2.add(Embedding(vocab, 64))
model_RNN2.add(SimpleRNN(32))
model_RNN2.add(Dropout(0.5))
model_RNN2.add(Dense(1, activation='sigmoid'))
model_RNN2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_RNN2.summary()

start = time.time()
history_RNN2 = model_RNN2.fit(X_train_vec, Y_train, epochs=10, batch_size=32,
                              validation_data=(X_test_vec, Y_test),
                              class_weight={0: 1, 1: 5}, verbose=0)
time2 = time.time() - start

record('노드 32개 / 층 1개', 32, 1, model_RNN2, time2)
Eval_func(model_RNN2, history_RNN2)

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_16 (Embedding)        │ (None, 48, 64)         │        72,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_23 (SimpleRNN)       │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,649 (295.50 KB)

 Trainable params: 75,649 (295.50 KB)

 Non-trainable params: 0 (0.00 B)

{'조건': '노드 32개 / 층 1개', '노드수': 32, '층수': 1, '파라미터수': 75649, '학습시간(초)': 21.3, '학습정확도': 0.946, '검증정확도': 0.942, 'F1 스코어': 0.824}
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      2666
           1       0.82      0.85      0.83       510

    accuracy                           0.95      3176
   macro avg       0.90      0.91      0.90      3176
weighted avg       0.95      0.95      0.95      3176

              precision    recall  f1-score   support

           0       0.97      0.96      0.97       666
           1       0.81      0.84      0.82       128

    accuracy                           0.94       794
   macro avg       0.89      0.90      0.89       794
weighted avg       0.94      0.94      0.94       794



In [57]:
# 실험 3) 노드 128개 / 1층
tf.keras.utils.set_random_seed(1234)

model_RNN3 = Sequential()
model_RNN3.add(Input(shape=(max_len,)))
model_RNN3.add(Embedding(vocab, 64))
model_RNN3.add(SimpleRNN(128))
model_RNN3.add(Dropout(0.5))
model_RNN3.add(Dense(1, activation='sigmoid'))
model_RNN3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_RNN3.summary()

start = time.time()
history_RNN3 = model_RNN3.fit(X_train_vec, Y_train, epochs=10, batch_size=32,
                              validation_data=(X_test_vec, Y_test),
                              class_weight={0: 1, 1: 5}, verbose=0)
time3 = time.time() - start

record('노드 128개', 128, 1, model_RNN3, time3)
Eval_func(model_RNN3, history_RNN3)

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_17 (Embedding)        │ (None, 48, 64)         │        72,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_24 (SimpleRNN)       │ (None, 128)            │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 97,345 (380.25 KB)

 Trainable params: 97,345 (380.25 KB)

 Non-trainable params: 0 (0.00 B)

{'조건': '노드 128개', '노드수': 128, '층수': 1, '파라미터수': 97345, '학습시간(초)': 27.7, '학습정확도': 0.905, '검증정확도': 0.851, 'F1 스코어': 0.638}
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

           0       0.98      0.90      0.94      2666
           1       0.64      0.92      0.76       510

    accuracy                           0.90      3176
   macro avg       0.81      0.91      0.85      3176
weighted avg       0.93      0.90      0.91      3176

              precision    recall  f1-score   support

           0       0.96      0.86      0.91       666
           1       0.53      0.81      0.64       128

    accuracy                           0.85       794
   macro avg       0.74      0.84      0.77       794
weighted avg       0.89      0.85      0.86       794



In [58]:
# 실험 4) 노드 32개 / 2층
tf.keras.utils.set_random_seed(1234)

model_RNN4 = Sequential()
model_RNN4.add(Input(shape=(max_len,)))
model_RNN4.add(Embedding(vocab, 64))
model_RNN4.add(SimpleRNN(32, return_sequences=True))
model_RNN4.add(SimpleRNN(32))
model_RNN4.add(Dropout(0.5))
model_RNN4.add(Dense(1, activation='sigmoid'))
model_RNN4.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_RNN4.summary()

start = time.time()
history_RNN4 = model_RNN4.fit(X_train_vec, Y_train, epochs=10, batch_size=32,
                              validation_data=(X_test_vec, Y_test),
                              class_weight={0: 1, 1: 5}, verbose=0)
time4 = time.time() - start

record('층 2개', 32, 2, model_RNN4, time4)
Eval_func(model_RNN4, history_RNN4)

Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_18 (Embedding)        │ (None, 48, 64)         │        72,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_25 (SimpleRNN)       │ (None, 48, 32)         │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_26 (SimpleRNN)       │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,729 (303.63 KB)

 Trainable params: 77,729 (303.63 KB)

 Non-trainable params: 0 (0.00 B)

{'조건': '층 2개', '노드수': 32, '층수': 2, '파라미터수': 77729, '학습시간(초)': 38.3, '학습정확도': 0.94, '검증정확도': 0.903, 'F1 스코어': 0.746}
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
              precision    recall  f1-score   support

           0       0.99      0.94      0.96      2666
           1       0.74      0.95      0.84       510

    accuracy                           0.94      3176
   macro avg       0.87      0.95      0.90      3176
weighted avg       0.95      0.94      0.94      3176

              precision    recall  f1-score   support

           0       0.98      0.91      0.94       666
           1       0.65      0.88      0.75       128

    accuracy                           0.90       794
   macro avg       0.81      0.89      0.84       794
weighted avg       0.92      0.90      0.91       794



In [59]:
# 실험 5) 노드 32개 / 3층
tf.keras.utils.set_random_seed(1234)

model_RNN5 = Sequential()
model_RNN5.add(Input(shape=(max_len,)))
model_RNN5.add(Embedding(vocab, 64))
model_RNN5.add(SimpleRNN(32, return_sequences=True))
model_RNN5.add(SimpleRNN(32, return_sequences=True))
model_RNN5.add(SimpleRNN(32))
model_RNN5.add(Dropout(0.5))
model_RNN5.add(Dense(1, activation='sigmoid'))
model_RNN5.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_RNN5.summary()

start = time.time()
history_RNN5 = model_RNN5.fit(X_train_vec, Y_train, epochs=10, batch_size=32,
                              validation_data=(X_test_vec, Y_test),
                              class_weight={0: 1, 1: 5}, verbose=0)
time5 = time.time() - start

record('층 3개', 32, 3, model_RNN5, time5)
Eval_func(model_RNN5, history_RNN5)

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_19 (Embedding)        │ (None, 48, 64)         │        72,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_27 (SimpleRNN)       │ (None, 48, 32)         │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_28 (SimpleRNN)       │ (None, 48, 32)         │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_29 (SimpleRNN)       │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,809 (311.75 KB)

 Trainable params: 79,809 (311.75 KB)

 Non-trainable params: 0 (0.00 B)

{'조건': '층 3개', '노드수': 32, '층수': 3, '파라미터수': 79809, '학습시간(초)': 41.3, '학습정확도': 0.934, '검증정확도': 0.904, 'F1 스코어': 0.75}
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
              precision    recall  f1-score   support

           0       0.99      0.93      0.96      2666
           1       0.72      0.95      0.82       510

    accuracy                           0.93      3176
   macro avg       0.86      0.94      0.89      3176
weighted avg       0.95      0.93      0.94      3176

              precision    recall  f1-score   support

           0       0.98      0.91      0.94       666
           1       0.65      0.89      0.75       128

    accuracy                           0.90       794
   macro avg       0.81      0.90      0.85       794
weighted avg       0.92      0.90      0.91       794



In [54]:
# 전체 결과 비교
result_df = pd.DataFrame(results)
display(result_df)
print()
print('=== 노드 수 비교 (층 1개 고정) ===')
display(result_df[result_df['층수'] == 1])
print()
print('=== 층 수 비교 (노드 32개 고정) ===')
display(result_df[result_df['노드수'] == 32])

fig = px.bar(result_df, x='조건', y=['학습정확도', '검증정확도', 'F1 스코어'],
             barmode='group', title='RNN 하이퍼파라미터별 성능 비교')
fig.show()

,조건,노드수,층수,파라미터수,학습시간(초),학습정확도,검증정확도,F1 스코어
0,노드 8개,8,1,73105,18.3,0.910,0.882,0.706
1,노드 32개 / 층 1개,32,1,75649,27.9,0.946,0.942,0.824
2,노드 128개,128,1,97345,29.6,0.905,0.851,0.638
3,층 2개,32,2,77729,29.6,0.940,0.903,0.746
4,층 3개,32,3,79809,43.8,0.934,0.904,0.750



=== 노드 수 비교 (층 1개 고정) ===


,조건,노드수,층수,파라미터수,학습시간(초),학습정확도,검증정확도,F1 스코어
0,노드 8개,8,1,73105,18.3,0.910,0.882,0.706
1,노드 32개 / 층 1개,32,1,75649,27.9,0.946,0.942,0.824
2,노드 128개,128,1,97345,29.6,0.905,0.851,0.638



=== 층 수 비교 (노드 32개 고정) ===


,조건,노드수,층수,파라미터수,학습시간(초),학습정확도,검증정확도,F1 스코어
1,노드 32개 / 층 1개,32,1,75649,27.9,0.946,0.942,0.824
3,층 2개,32,2,77729,29.6,0.940,0.903,0.746
4,층 3개,32,3,79809,43.8,0.934,0.904,0.750


In [62]:
# 실험별 Embedding 비중 정리
models = {'노드 8개(1층)': model_RNN1, '노드 32개(1층)': model_RNN2, '노드 128개(1층)': model_RNN3,
          '노드 32개(2층)': model_RNN4, '노드 32개(3층)': model_RNN5}

rows = []
for name, m in models.items():
    emb = m.layers[0].count_params()           # Embedding 파라미터
    total = m.count_params()
    rows.append({'조건': name, 'Embedding': emb, 'RNN+Dense': total - emb,
                 '전체': total, 'Embedding비중': f'{emb/total:.1%}'})

pd.DataFrame(rows)

,조건,Embedding,RNN+Dense,전체,Embedding비중
0,노드 8개(1층),72512,593,73105,99.2%
1,노드 32개(1층),72512,3137,75649,95.9%
2,노드 128개(1층),72512,24833,97345,74.5%
3,노드 32개(2층),72512,5217,77729,93.3%
4,노드 32개(3층),72512,7297,79809,90.9%


- 층 추가 : 층당 정확히 2,080개씩 늘어남.
    - 75,649 → 77,729 → 79,809
    - 전체 파라미터의 74~99%가 Embedding - 노드·층을 조정해도 모델 크기 자체는 크게 변하지 않음

- 노드 수 비교 (1층 고정) : **32개가 최적**
| 노드 | 파라미터 | 시간 | 학습정확도 | 검증정확도 | 차이 | F1 |
| -- | -- | -- | -- | -- | -- | -- |
| 8 | 73,105 | 18.3초 | 0.910 | 0.882 | 0.028 | 0.706 |
| 32 | 75,649 | 27.9초 | 0.946 | 0.942 | 0.004 | 0.824 |
| 128 | 97,345 | 29.6초 | 0.905 | 0.851 | 0.054 | 0.638 |

  - 노드 8개: 학습·검증 정확도가 모두 낮음.
  - 노드 32개: 학습(0.946)과 검증(0.942) 차이가 0.004로 가장 작음. F1 스코어도 0.824로 가장 높음
  - 노드 128개: 학습 정확도까지 떨어짐(0.905). 검증-학습 차이(0.054)도 가장 큼.

- 층 수 비교 (노드 32개 고정) : **층을 쌓을수록 나빠짐**
| 층 | 파라미터 | 시간 | 학습정확도 | 검증정확도 | F1 |
| 1 | 75,649 | 27.9초 | 0.946 | 0.942 | 0.824 |
| 2 | 77,729 | 29.6초 | 0.940 | 0.903 | 0.746 |
| 3 | 79,809 | 43.8초 | 0.934 | 0.904 | 0.750 |

- 파라미터는 2.7%만 늘었는데 F1은 9.5% 하락 (비용 대비 손해)
- 학습 정확도도 함께 하락(0.946 → 0.934).
- 위 노드 128개와 같은 현상으로, 층이 깊어질수록 기울기가 앞쪽 층까지 제대로 전달되지 않는 SimpleRNN의 한계로 보임
- 학습 시간은 3층에서 43.8초로 57% 증가
  - 층은 순차적으로 계산해야 해서 병렬화가 안 되기 때문
  - 노드를 8 → 128로 16배 늘렸을 때 시간이 18.3 → 29.6초로 1.6배만 늘어난 것과 대비됨

#### **최적 구조 : SimpleRNN 노드 32개, 1층**

- 검증 정확도 0.942, F1 0.824, 학습 시간 27.9초

---


- 모델을 키운다고 성능이 좋아지지 않는다
  - 데이터 3,970건, 어휘 1,133개, 문장 평균 17단어로 규모가 작음.
  - 노드 128개나 3층 구조는 이 데이터에 비해 용량이 과해서, 학습이 수렴하지 못하고 오히려 성능이 떨어짐.
  - 데이터 규모에 맞는 모델 크기를 고르는 것이 크기를 키우는 것보다 중요.




---


- 모델의 숫자 대부분(74~99%)은 단어 뜻을 담는 Embedding이 차지하고 이 부분은 항상 고정되므로,  
노드를 16배 늘려도 전체 파라미터는 1.3배밖에 늘지 않는다.